# 01 — Data loading and exploratory analysis

First of two notebooks. This one **loads, inspects and characterises** the raw data;
notebook `02` does the cleaning and feature engineering and reads the parquet
extracts written here into `notebooks/data/`.

## What the data is

* **Meter readings** — M1 smart meters, i.e. the meter sitting at the *grid
  connection point*, so what it records is the **exchange with the grid**
  (import from / export to), not household consumption or PV generation as such.
  One row per device per **15-minute** interval.
* **mySET readings** — the same 15-minute grid-exchange quantities for **five**
  of those devices, but taken from the **mySET** portal of SET Distribuzione, the
  local distribution operator. mySET publishes the readings of the **DSO's own meter
  at the POD** — the same grid connection point as the M1 meter — as kWh
  **imported** from and **exported** to the grid per quarter-hour. It is the only
  source that holds those five devices' history before 2026-05-21.
* **Weather** — hourly features for the meters' area (Folgaria, Trentino),
  precomputed by an upstream pipeline into the gold layer, and complemented here
  with a direct Open-Meteo download using the **same model as the pipeline
  (ICON-D2)**.

## The units trap

The columns named `*_kw` / `*_kwh` all hold **energy in kWh accumulated over one
15-minute interval** — *not* instantaneous power. Consequences:

* hourly values are the **SUM of the four quarters**, never their mean;
* a value of `0.25` means 0.25 kWh in 15 min, i.e. an average of 1 kW over that quarter;
* if a source ever reports average power in kW, it must be multiplied by `0.25` first.

The `_kw` suffix in the database is legacy. See `docs/data_contract.md`.

## Counting frames (`cf_type`)

The meter emits readings under several *counting frames*:

| frame | meaning |
|---|---|
| `CF1`, `CF3` | consumption-only frames (monophase / triphase) |
| `CF2`, `CF4` | frames carrying both consumption and production |
| `CF101`, `CF103` | **M2** frames — the PV-side meter, a different measurement point |

Only `M1` + `CF1..CF4` belong to the grid-exchange series analysed here; the M2
frames are excluded. This mirrors the repo's `datasets.local.yaml`, so the notebook
loads exactly what the CLI loads.

## Where the data comes from, and how to run this

All three tables — the MQTT meter readings, the mySET export and the gold-layer
weather — live in the same **dev PostgreSQL** database, reached through a local
tunnel on `localhost:25432`. Credentials are **never** written in this notebook:
they come from the gitignored `.env` via `pydantic-settings`
(`celine.forecasting.core.settings.settings.database_url`).

```bash
# 1. dependencies
uv sync --extra db --extra notebooks

# 2. tunnel (leave it running in another terminal)
kubectl --context k8s-pve -n celine-staging port-forward svc/postgis1-rw 25432:5432

# 3. this notebook
uv run jupyter lab notebooks/01_data_loading_and_eda.ipynb
```

The tunnel is wrapped in a restart loop and occasionally drops for a couple of
seconds. If a query fails with a connection error, wait a moment and re-run the cell.

## 1. Setup

Imports, the database engine and the paths.

`DATA_DIR` is `data/` **next to this notebook** (`notebooks/data/`), which the
repository's root `data/` ignore rule already covers — nothing extracted here can
be committed by accident. The working directory is moved to the repository root
so that `pydantic-settings` finds the gitignored `.env`; no path is hardcoded.

In [ ]:
import os
import time
import warnings
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sqlalchemy as sa

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)

# The extracts are written next to this notebook, in notebooks/data/.
NB_DIR = Path.cwd().resolve()

# Walk up from the notebook folder until we find the folder that holds pyproject.toml.
REPO_ROOT = None
for folder in [NB_DIR] + list(NB_DIR.parents):
    if (folder / "pyproject.toml").exists():
        REPO_ROOT = folder
        break
if REPO_ROOT is None:
    raise FileNotFoundError("could not find pyproject.toml above the notebook folder")

# pydantic-settings reads .env from the working directory, so work from the repo root.
os.chdir(REPO_ROOT)

DATA_DIR = NB_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from celine.forecasting.core.db import load_meters_from_db, load_weather_from_db
from celine.forecasting.core.validation import validate_raw_schema
from celine.forecasting.core.settings import settings

# The connection string is never written in the notebook: pydantic-settings reads it
# from the DATABASE_URL entry of the gitignored .env at the repository root.
DATABASE_URL = settings.database_url
engine = sa.create_engine(DATABASE_URL)

METERS_TABLE = "ds_dev_silver.meters_data"
WEATHER_TABLE = "ds_dev_gold.om_weather_features_meters"
LOCAL_TZ = "Europe/Rome"

# The mySET export of the DSO meter at the POD. Its `ts` is NOT UTC: it is the local
# (Europe/Rome) wall clock, and it labels a quarter by its START, where the MQTT meter
# labels it by its END — both established in section 4.
MYSET_TABLE = "ds_dev_silver.silver_myset_quarterly"

# The five May-2026 devices whose earlier history exists only in the mySET export.
MYSET_DEVICES = [
    "c2g-9FFB89CF4",
    "c2g-9FFB89ED4",
    "c2g-9FFB8A0D0",
    "c2g-9FFB8AA78",
    "c2g-DD6C2E6EC",
]

# Site of the meters (Folgaria, Trentino) — used for the Open-Meteo complement.
# These are the coordinates the upstream tap sends to Open-Meteo.
SITE_LAT, SITE_LON = 45.9167, 11.1667

# Same model as the upstream tap, which extracts with models=icon_d2
# (DWD ICON-D2, 2.2 km). There is deliberately NO elevation override here: the tap
# sends none either, so Open-Meteo resolves the site on its own 90 m DEM, which
# returns 1172 m. Forcing elevation=1100 makes Open-Meteo downscale temperature by
# lapse rate over those 72 m, about 0.47 °C — the constant 0.45 °C offset against
# the gold table that this notebook used to show.
WEATHER_MODEL = "icon_d2"

## 2. Meters — inventory straight from SQL

Before loading anything into memory, ask the database what is in the table. Three
cheap aggregate queries:

1. the full `meter_type` x `cf_type` breakdown — this is what justifies the
   `M1` + `CF1..CF4` filter;
2. a per-device, per-frame inventory (row counts, first/last timestamp, mean
   consumption and production);
3. the `(device_id, ts)` pairs that appear twice — which turn out **not** to be a
   frame problem at all.

In [ ]:
# Full meter_type x cf_type breakdown — this is what justifies the M1 + CF1..CF4 filter.
breakdown = pd.read_sql(f"""
    SELECT meter_type, cf_type, COUNT(*) AS rows,
           MIN(ts) AS first_ts, MAX(ts) AS last_ts,
           COUNT(DISTINCT device_id) AS devices
    FROM {METERS_TABLE}
    GROUP BY meter_type, cf_type
    ORDER BY meter_type, cf_type
""", engine)
breakdown

The M1 grid-exchange rows are the four `CF1..CF4` frames. Everything else is a
different measurement point (M2 / M2_2, the PV-side meters) and is dropped.

Two oddities are visible above and worth carrying forward. The silver model copies
`cf_type` from the MQTT payload's `Type` field and `meter_type` from its `Meter`
field, both verbatim, so a message whose two fields disagree arrives mislabelled:

* fifteen `M2` rows carrying `CF2`/`CF4` — these are **real M1 readings wearing the
  wrong `Meter` label**. The inconsistent pair comes off the device/gateway itself
  (the raw MQTT message already carries it), and the `consumption_kwh` in those rows
  is the genuine M1 grid-import value for that quarter-hour: on 2026-06-10 12:30 the
  stray row reads 1.809 kWh, between M1 neighbours of 2.009 (12:15) and 1.880 (12:45).
  Rate in the retained raw window: 2 of about 13,000 `CF2` messages;
* a **single** `M1` row carrying the M2 frame `CF101` — the mirror case.

The cost is larger than the sixteen excluded rows suggest. The silver dedup runs
`row_number() over (partition by device_id, ts, meter_type order by _id desc)`, so a
mislabelled message lands in the `M2` partition and **evicts the genuine `CF101` PV
reading** for that quarter. At each of those fifteen instants both the M1 row and the
true M2 row are therefore absent: the filter loses one grid reading *and* one PV
reading per event. The frames table above is right as it stands — M2 (`CF101`/`CF103`)
is production-only.

Note also that each device uses exactly **one** counting frame for its whole history
(see the next cell), so a device never straddles two frames.

In [ ]:
# The rows whose measurement point and counting frame disagree, i.e. what the filter drops.
stray = pd.read_sql(f"""
    SELECT device_id, meter_type, cf_type, ts, consumption_kwh, production_kwh
    FROM {METERS_TABLE}
    WHERE (meter_type = 'M1' AND cf_type NOT IN ('CF1','CF2','CF3','CF4'))
       OR (meter_type <> 'M1' AND cf_type IN ('CF1','CF2','CF3','CF4'))
    ORDER BY meter_type, ts
""", engine)
print("cross-labelled rows excluded by the filter:", len(stray))
stray

In [ ]:
# Per-device, per-frame inventory: counts, span and mean values, computed by the server.
inventory = pd.read_sql(f"""
    SELECT device_id, cf_type,
           COUNT(*)                      AS rows,
           MIN(ts)                       AS first_ts,
           MAX(ts)                       AS last_ts,
           AVG(consumption_kwh)          AS mean_cons_kwh_15min,
           AVG(production_kwh)           AS mean_prod_kwh_15min
    FROM {METERS_TABLE}
    WHERE meter_type = 'M1' AND cf_type IN ('CF1','CF2','CF3','CF4')
    GROUP BY device_id, cf_type
    ORDER BY first_ts, device_id, cf_type
""", engine)

total_rows = inventory["rows"].sum()
print("M1 devices:", inventory["device_id"].nunique())
print("(device, frame) combinations:", len(inventory))
print(f"rows: {total_rows:,}")

# How many distinct counting frames each device uses over its whole history.
frames_per_device = inventory.groupby("device_id")["cf_type"].nunique()
print("\ndevices per number of frames:")
print(frames_per_device.value_counts())

print("\nrows per frame:")
print(inventory.groupby("cf_type")["rows"].sum())
inventory.head(12)

In [ ]:
# The (device_id, ts) pairs that appear more than once, with the frames involved.
dupes = pd.read_sql(f"""
    SELECT device_id, ts, COUNT(*) AS n_rows,
           COUNT(DISTINCT cf_type) AS n_frames,
           STRING_AGG(DISTINCT cf_type, '+' ORDER BY cf_type) AS frames
    FROM {METERS_TABLE}
    WHERE meter_type = 'M1' AND cf_type IN ('CF1','CF2','CF3','CF4')
    GROUP BY device_id, ts
    HAVING COUNT(*) > 1
    ORDER BY ts
""", engine)

# The explanation lives on the local wall clock, so convert the instants.
dupes["ts_local"] = dupes["ts"].dt.tz_convert(LOCAL_TZ)

print("duplicated (device_id, ts) pairs:", len(dupes))
# One frame per pair means the duplicates are within a frame, not across frames.
print("distinct frames involved per pair:", sorted(dupes["n_frames"].unique()))
print("distinct UTC instants:", dupes["ts"].nunique())
print("first:", dupes["ts"].min(), " last:", dupes["ts"].max())
print("devices affected:", dupes["device_id"].nunique())

local_times = set()
for timestamp in dupes["ts_local"]:
    local_times.add(str(timestamp))
print("local times of the duplicated instants:", sorted(local_times))
dupes.head(8)

**These are not frame duplicates.** All 36 sit in the same four quarter-hours of
**2025-10-26**, they involve a single counting frame each, and they hit exactly the
nine devices that were already online at that date. That is the night European summer
time ends: the local wall clock runs 02:00–02:59 twice. The upstream dedup key is
`(device_id, ts, meter_type)` on a *local* timestamp, so the repeated wall-clock hour
produces two rows that map to two different UTC instants — and the ambiguity lands
here.

The package loader drops them with `keep="first"` on `(device_id, ts)`, so 36 readings
(9 devices x 4 quarters, one autumn night) are discarded. Harmless at this scale, but
notebook 02 should confirm the *spring-forward* counterpart does not instead create a
one-hour hole.

## 3. Meters — load through the package loader

We deliberately do **not** write our own SQL here: `load_meters_from_db` is the same
code path the CLI uses (`datasets.local.yaml`), so whatever the notebook sees is what
the pipeline sees.

Note the `columns` map: it renames `consumption_kwh -> consumption_kw`, then
`normalize_meters` maps `consumption_kw` back to `consumption_kwh` through its alias
table. Round trip, no net effect — but it is kept **identical to the repo config** on
purpose.

In [ ]:
METERS_SOURCES = [{
    "table": METERS_TABLE,
    "filters": {"meter_type": "M1", "cf_type": ["CF1", "CF2", "CF3", "CF4"]},
    "columns": {"consumption_kwh": "consumption_kw", "production_kwh": "production_kw"},
}]

# Load through the package loader, the same code path the CLI uses.
df_meters = load_meters_from_db(METERS_SOURCES, engine=engine)

memory_mb = df_meters.memory_usage(deep=True).sum() / 1e6
print("shape:", df_meters.shape)
print(f"memory: {memory_mb:.1f} MB")
print("rows dropped as (device_id, ts) duplicates:", inventory["rows"].sum() - len(df_meters))
print()
print(df_meters.dtypes)
df_meters.head()

The loader keeps the extra columns it does not need (`_id`, `cf_type`, `meter_type`),
so the per-frame provenance of every reading survives into the extract — handy for
notebook 02.

In [ ]:
# Are the timestamps on a clean 15-minute grid?
print("readings per minute of the hour:")
print(df_meters["ts"].dt.minute.value_counts().sort_index())

print("\nseconds:", df_meters["ts"].dt.second.unique())
print("microseconds:", df_meters["ts"].dt.microsecond.unique())
print("duplicates on (device_id, ts) after load:",
      int(df_meters.duplicated(subset=["device_id", "ts"]).sum()))

print("\nnulls per column:")
print(df_meters[["device_id", "ts", "consumption_kwh", "production_kwh"]].isna().sum())

# Negative values would be impossible for energy accumulated over an interval.
print("\nnegative consumption:", int((df_meters["consumption_kwh"] < 0).sum()))
print("negative production:", int((df_meters["production_kwh"] < 0).sum()))
print(f"max per 15 min: consumption {df_meters['consumption_kwh'].max():.3f} kWh, "
      f"production {df_meters['production_kwh'].max():.3f} kWh")

## 4. Five devices with a longer past — the mySET export

Five of the devices above — `c2g-9FFB89CF4`, `c2g-9FFB89ED4`, `c2g-9FFB8A0D0`,
`c2g-9FFB8AA78`, `c2g-DD6C2E6EC` — appear in the MQTT table only from **2026-05-21**,
in the big onboarding wave. Nothing else in the database holds their earlier readings:
`ds_dev_gold.meters_data_15m`, `ds_dev_gold.meters_data_1h` and
`ds_dev_silver.rec_meters_15m` all start on that same date for them. But their history
does exist, in a second source.

**mySET** is the customer portal of *SET Distribuzione*, the local distribution system
operator. What it publishes is the reading of the **DSO's own meter at the POD** — the
same grid connection point the M1 meter sits on, measured by a different device and
delivered by a different route. The route is:

1. meltano's `tap-myset-data` (a `tap-jsonfile`) reads the portal's JSON exports from
   `s3://$MYSET_S3_BUCKET/*/*/*/*.json` into `raw.myset_data`, keeping `imported` and
   `exported` as `jsonb`;
2. `dbt/models/staging/myset/stg_myset_data.sql` explodes `data.consumptions[]` — one
   entry per hour, carrying `year`, `month`, `day`, `hour`, `estimated` and a
   `quarters` array of four values — into one row per quarter, with
   `make_timestamp(year, month, day, hour, (quarter_index - 1) * 15, 0)`. That is a
   **naive** timestamp: no timezone is attached anywhere. `sensor_reference` is parsed
   out of the S3 file path, and `exported` is left-joined and coalesced to 0;
3. `dbt/models/silver/myset/silver_myset_quarterly.sql` dedups on
   `(sensor_reference, ts)` keeping the latest `_sdc_extracted_at`.

Two things have to be settled before any of it can be merged into `df_meters`:

* **what clock `ts` is on** — the staging model attaches no timezone, so the column is
  either UTC or the local wall clock, and nothing in the schema says which;
* **how its quarters line up with the MQTT readings** — even on the right clock, two
  metering systems can label the same quarter-hour by its start or by its end.

Both are answered below from the data itself, on the 2026-05-21 → 2026-06-08 window
where the two sources overlap.


In [ ]:
# Server-side inventory of the mySET export, one row per sensor_reference.
myset_inventory = pd.read_sql(sa.text(f"""
    SELECT sensor_reference,
           COUNT(*)                                   AS rows,
           COUNT(DISTINCT ts)                         AS distinct_ts,
           MIN(ts)                                    AS first_ts,
           MAX(ts)                                    AS last_ts,
           AVG(estimated::int)                        AS share_estimated,
           COUNT(*) FILTER (WHERE imported_kwh IS NULL
                              OR exported_kwh IS NULL) AS nulls,
           COUNT(*) FILTER (WHERE imported_kwh < 0
                              OR exported_kwh < 0)     AS negatives,
           MAX(imported_kwh)                          AS max_imported_kwh,
           MAX(exported_kwh)                          AS max_exported_kwh,
           COUNT(*) FILTER (WHERE ts < '2026-01-01')   AS rows_2025,
           COUNT(*) FILTER (WHERE ts >= '2026-01-01')  AS rows_2026
    FROM {MYSET_TABLE}
    GROUP BY sensor_reference
    ORDER BY sensor_reference
"""), engine)

print("sensor_reference values in the table:", len(myset_inventory))
print("all five devices of interest present:",
      sorted(myset_inventory["sensor_reference"]) == sorted(MYSET_DEVICES))
print(f"rows: {myset_inventory['rows'].sum():,}")
print("minutes of the hour used:")
print(pd.read_sql(sa.text(f"""
    SELECT EXTRACT(MINUTE FROM ts)::int AS minute, COUNT(*) AS rows
    FROM {MYSET_TABLE} GROUP BY 1 ORDER BY 1
"""), engine).to_string(index=False))

# The three days European summer time starts or ends inside the export's span.
dst_days = pd.read_sql(sa.text(f"""
    SELECT sensor_reference, ts::date AS day, COUNT(*) AS rows
    FROM {MYSET_TABLE}
    WHERE ts::date::text = ANY(:days)
    GROUP BY sensor_reference, ts::date
    ORDER BY day, sensor_reference
"""), engine, params={"days": ["2025-03-30", "2025-10-26", "2026-03-29"]})
print("\nrows per device on the two DST days (96 = a full day):")
print(dst_days.pivot(index="sensor_reference", columns="day", values="rows"))

myset_inventory


The export is **clean and complete**: 240,208 rows over five `sensor_reference`
values, which are exactly the five devices of interest; no nulls, no negatives, the
minutes of the hour are exactly 0 / 15 / 30 / 45 in equal counts, and `estimated` is
`False` on every single row — nothing in this table is an upstream estimate. Three
devices start on **2025-01-01 00:00**, `c2g-9FFB89ED4` on **2025-02-24 10:00** and
`c2g-9FFB8AA78` on **2025-03-05 00:00**; all five stop at **2026-06-08 23:45**.

The interesting number is the daily row count. Every device delivers 96 rows a day —
except on **2025-03-30** and **2026-03-29**, where it delivers **92**, and on
**2025-10-26**, where it still delivers **96**. Those are precisely the two spring-forward
nights and the one fall-back night of the span. A UTC clock has 96 quarters on every
day of the year, so this pattern alone settles the first question:

* **92 rows** on the spring-forward days — the local clock jumps 01:45 → 03:00, and the
  four quarters of the skipped 02:xx hour simply do not exist. These are the only steps
  larger than 15 minutes anywhere in the table;
* **96 rows** on the fall-back day — the local clock runs 02:00–02:59 *twice*, so the
  wall clock has 100 quarters that day, but the silver dedup key is
  `(sensor_reference, ts)` on the naive stamp, so the second copy of each repeated
  quarter **overwrites** the first. Four readings per device are lost upstream, and the
  table cannot tell us which copy survived.

So `ts` is the **local (Europe/Rome) wall clock**, with the repeated hour collapsed.
(`c2g-9FFB89ED4` also shows one short day, 56 rows: its first day, 2025-02-24, which
starts at 10:00.)


In [ ]:
# The five devices' whole mySET history, read with a parameterised query.
myset_query = sa.text(f"""
    SELECT _id, sensor_reference, ts, imported_kwh, exported_kwh
    FROM {MYSET_TABLE}
    WHERE sensor_reference = ANY(:devices)
    ORDER BY sensor_reference, ts
""")
with engine.connect() as conn:
    myset_raw = pd.read_sql(myset_query, conn, params={"devices": MYSET_DEVICES})
print("rows read:", f"{len(myset_raw):,}")

# Normalise through the package, exactly as the CLI loader does for a source with
# `assume_tz` set: rename to the contract names and localise the naive stamps as
# Europe/Rome before converting to UTC.
from celine.forecasting.core.ingest import normalize_meters

myset = normalize_meters(
    myset_raw,
    assume_tz=LOCAL_TZ,
    column_map={
        "sensor_reference": "device_id",
        "imported_kwh": "consumption_kwh",
        "exported_kwh": "production_kwh",
    },
)

# `ambiguous="NaT"` turns the repeated fall-back hour into NaT, because the export
# holds only one copy of it and pandas cannot know which one it is.
is_nat = myset["ts"].isna()
print("rows dropped as ambiguous local timestamps (NaT):", int(is_nat.sum()))
nat_local = sorted(set(myset_raw.loc[is_nat.values, "ts"].astype(str)))
print("local instants they were:", nat_local)
myset = myset[~is_nat].reset_index(drop=True)

# The MQTT readings of the same five devices, on the window the two sources share.
mqtt_five = df_meters[df_meters["device_id"].isin(MYSET_DEVICES)]
mqtt_five = mqtt_five[["device_id", "ts", "consumption_kwh", "production_kwh"]]
overlap_start = mqtt_five["ts"].min()
overlap_end = myset["ts"].max()
print(f"\noverlap window (UTC): {overlap_start} -> {overlap_end}")

in_overlap = (mqtt_five["ts"] >= overlap_start) & (mqtt_five["ts"] <= overlap_end)
mqtt_overlap = mqtt_five[in_overlap]
print(f"MQTT quarters in the overlap: {len(mqtt_overlap):,}")


def safe_corr(x, y):
    """Pearson r, or NaN when one of the series is constant (flat zero export)."""
    if x.nunique() < 2 or y.nunique() < 2:
        return np.nan
    return round(x.corr(y), 3)


def agreement(mqtt_frame, myset_frame, shift):
    """Join the two sources quarter by quarter with `shift` applied to mySET."""
    shifted = myset_frame[["device_id", "ts", "consumption_kwh", "production_kwh"]].copy()
    shifted["ts"] = shifted["ts"] + pd.Timedelta(minutes=shift)
    return mqtt_frame.merge(shifted, on=["device_id", "ts"], how="inner",
                            suffixes=("_mqtt", "_myset"))


# ALIGNMENT SCAN: which offset makes the two meters describe the same quarter?
scan_rows = []
for shift in [-30, -15, 0, 15, 30, 45, 60]:
    j = agreement(mqtt_overlap, myset, shift)
    if j.empty:
        continue
    import_diff = (j["consumption_kwh_mqtt"] - j["consumption_kwh_myset"]).abs()
    export_diff = (j["production_kwh_mqtt"] - j["production_kwh_myset"]).abs()
    scan_rows.append({
        "shift_min": shift,
        "matched_quarters": len(j),
        "corr_import": safe_corr(j["consumption_kwh_mqtt"], j["consumption_kwh_myset"]),
        "import_within_1.5Wh": round((import_diff < 0.0015).mean(), 3),
        "corr_export": safe_corr(j["production_kwh_mqtt"], j["production_kwh_myset"]),
        "export_exact": round((export_diff < 1e-9).mean(), 3),
    })
scan = pd.DataFrame(scan_rows)
print("\nalignment scan — all five devices, whole overlap:")
print(scan.to_string(index=False))

# The offset that wins, and the same table split by device.
MYSET_QUARTER_SHIFT = pd.Timedelta(minutes=15)  # mySET labels a quarter by its START,
                                                # the MQTT meter by its END.
best_shift = int(MYSET_QUARTER_SHIFT.total_seconds() // 60)
print(f"\nper device at the winning shift (+{best_shift} min):")
j_best = agreement(mqtt_overlap, myset, best_shift)
per_device_rows = []
for device_id, g in j_best.groupby("device_id"):
    import_diff = (g["consumption_kwh_mqtt"] - g["consumption_kwh_myset"]).abs()
    export_diff = (g["production_kwh_mqtt"] - g["production_kwh_myset"]).abs()
    per_device_rows.append({
        "device_id": device_id,
        "matched_quarters": len(g),
        "corr_import": safe_corr(g["consumption_kwh_mqtt"], g["consumption_kwh_myset"]),
        "import_within_1.5Wh": round((import_diff < 0.0015).mean(), 3),
        "corr_export": safe_corr(g["production_kwh_mqtt"], g["production_kwh_myset"]),
        "export_exact": round((export_diff < 1e-9).mean(), 3),
    })
print(pd.DataFrame(per_device_rows).to_string(index=False))

# The competing hypothesis: read `ts` as UTC instead, and scan whole hours of lag.
myset_as_utc = myset_raw[~is_nat.values].copy()
myset_as_utc = myset_as_utc.rename(columns={"sensor_reference": "device_id",
                                            "imported_kwh": "consumption_kwh",
                                            "exported_kwh": "production_kwh"})
myset_as_utc["ts"] = myset_as_utc["ts"].dt.tz_localize("UTC")
hour_rows = []
for lag_hours in range(-3, 4):
    j = agreement(mqtt_overlap, myset_as_utc, lag_hours * 60 + best_shift)
    if j.empty:
        continue
    hour_rows.append({
        "lag_hours": lag_hours,
        "matched_quarters": len(j),
        "corr_import": safe_corr(j["consumption_kwh_mqtt"], j["consumption_kwh_myset"]),
    })
print(f"\nunder the 'ts is UTC' hypothesis, whole-hour correction "
      f"(the +{best_shift} min is kept):")
print(pd.DataFrame(hour_rows).to_string(index=False))

# The aligned mySET frame every cell below works with.
df_myset = myset.copy()
df_myset["ts"] = df_myset["ts"] + MYSET_QUARTER_SHIFT
print(f"\ndf_myset: {len(df_myset):,} rows, "
      f"{df_myset['ts'].min()} -> {df_myset['ts'].max()} (UTC)")


**Both questions are answered by the scan.** The `+15 min` row wins on every measure
at once — import correlation 0.922 against 0.880 at both neighbouring offsets, 72.5 %
of quarters agreeing to within 1.5 Wh against 34 %, and the best export figures too.
Note the shape of the table: 0 and +30 sit at exactly the same lower value as each
other, and so do −15 and +45, which is what a *symmetric* miss around a true offset
looks like. The cause is a labelling convention, not a clock error: mySET names a
quarter by the moment it **starts** (quarter *k* of hour *h* is stamped `h:(k-1)*15`),
the MQTT meter names it by the moment it **ends**. Adding 15 minutes to the mySET stamp
turns one into the other.

The clock reading survives the obvious alternative. If `ts` were UTC, no whole-hour
correction would be needed — but read that way it needs **−2 h** to reach the same
0.922, and −2 h is exactly the CEST offset in force over the whole overlap window. A
fixed −2 h is not a clock, though: it would be an hour wrong for the entire winter half
of the export. Localising as `Europe/Rome` gets both halves right, which is why the
naive stamps are read as local time rather than shifted by a constant.

The `normalize_meters` call above does that first half — `tz_localize("Europe/Rome",
ambiguous="NaT", nonexistent="shift_forward")` then `tz_convert("UTC")`, the same code
the CLI loader runs for a source with `assume_tz` set. The **+15 minutes is applied in
the notebook**, right after it, because the package loader has no way to express a
per-source stamp offset: `datasets.yaml` carries `assume_tz` but no `ts_offset`. Adding
one is a sensible follow-up for the package; it is deliberately not done here.

One consequence to carry forward: on **2025-10-26** the ambiguous 02:00–02:45 local
rows become `NaT` and are dropped, four per device, twenty in total. Because the export
only ever held *one* copy of the repeated hour, the resulting UTC series has a
**two-hour hole** (00:15 → 02:00 UTC) where the MQTT side of the same night loses only
one hour.


In [ ]:
# How well do the two sources agree, day by day, at the winning shift?
j = j_best.copy()
j["day"] = j["ts"].dt.tz_convert(LOCAL_TZ).dt.floor("D").dt.date

daily_rows = []
for day, g in j.groupby("day"):
    import_diff = (g["consumption_kwh_mqtt"] - g["consumption_kwh_myset"]).abs()
    daily_rows.append({
        "day": day,
        "quarters": len(g),
        "exact_share": round((import_diff < 0.0015).mean(), 3),
        "import_mqtt_kwh": round(g["consumption_kwh_mqtt"].sum(), 1),
        "import_myset_kwh": round(g["consumption_kwh_myset"].sum(), 1),
    })
daily_agreement = pd.DataFrame(daily_rows)
daily_agreement["myset_over_mqtt"] = (
    daily_agreement["import_myset_kwh"] / daily_agreement["import_mqtt_kwh"]).round(2)
print("agreement by day — import summed over the five devices:")
print(daily_agreement.to_string(index=False))

# Per device, on the stable part of the overlap only.
STABLE_FROM = pd.Timestamp("2026-05-22").date()
STABLE_TO = pd.Timestamp("2026-06-03").date()
is_stable = (j["day"] >= STABLE_FROM) & (j["day"] <= STABLE_TO)
stable = j[is_stable]

ratio_rows = []
for device_id, g in stable.groupby("device_id"):
    import_diff = (g["consumption_kwh_mqtt"] - g["consumption_kwh_myset"]).abs()
    import_mqtt = g["consumption_kwh_mqtt"].sum()
    import_myset = g["consumption_kwh_myset"].sum()
    export_mqtt = g["production_kwh_mqtt"].sum()
    export_myset = g["production_kwh_myset"].sum()
    ratio_rows.append({
        "device_id": device_id,
        "quarters": len(g),
        "exact_share": round((import_diff < 0.0015).mean(), 3),
        "import_mqtt_kwh": round(import_mqtt, 1),
        "import_myset_kwh": round(import_myset, 1),
        "import_ratio": round(import_myset / import_mqtt, 2) if import_mqtt > 0 else np.nan,
        "export_mqtt_kwh": round(export_mqtt, 1),
        "export_myset_kwh": round(export_myset, 1),
        "corr_import": safe_corr(g["consumption_kwh_mqtt"],
                                 g["consumption_kwh_myset"]),
        "export_ratio": round(export_myset / export_mqtt, 2) if export_mqtt > 0 else np.nan,
    })
print(f"\nper device over {STABLE_FROM} -> {STABLE_TO} (the stable part of the overlap):")
print(pd.DataFrame(ratio_rows).to_string(index=False))

# One week of the overlap for a consumption-only device, both sources on one axis.
example_device = "c2g-9FFB89CF4"
week_from = pd.Timestamp("2026-05-22", tz="UTC")
week_to = week_from + pd.Timedelta(days=7)
is_example = (j["device_id"] == example_device) & (j["ts"] >= week_from) & (j["ts"] < week_to)
ex = j[is_example].sort_values("ts")
ex_local = ex["ts"].dt.tz_convert(LOCAL_TZ)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ex_local, ex["consumption_kwh_mqtt"], color="tab:blue", label="MQTT meter")
ax.plot(ex_local, ex["consumption_kwh_myset"], color="tab:orange", linestyle="--",
        label=f"mySET, shifted +{best_shift} min")
ax.set_ylim(bottom=0)
ax.set_title(f"Grid import, same quarters, two sources — {example_device} "
             "(consumption only)")
ax.set_ylabel("kWh per 15 min")
ax.set_xlabel("local time (Europe/Rome)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%a %d"))
ax.legend()
plt.tight_layout()
plt.show()

print(f"{len(ex):,} quarters plotted")


Two different stories in those tables.

**The tail of the export is not trustworthy.** From **2026-06-04** to the last day of
the export, **2026-06-08**, the mySET daily import totals run at **1.7–2.0x** the MQTT
ones (2026-06-06: 152.3 kWh against 76.6) and the share of exactly-agreeing quarters
collapses from ~0.89 to **0.21–0.31**. Whatever produced the last few days of the JSON
export — a partial re-run, a double-counted file — those days cannot be used. The merge
below never touches them: it takes mySET only *before* each device's first MQTT
reading, so the whole overlap, tail included, is served by MQTT.

**The three consumption-only devices agree to the Wh.** Over the stable stretch
(2026-05-22 → 2026-06-03) `c2g-9FFB89CF4` (CF3), `c2g-9FFB89ED4` (CF1) and
`c2g-DD6C2E6EC` (CF1) match on **100 %, 99.9 % and 100 %** of quarters respectively,
and their twelve-day import totals are identical to the printed 0.1 kWh (862.0, 131.8
and 66.2 kWh on both sides). Two independent meters at the same POD, agreeing quarter
by quarter for twelve days: that is as strong a confirmation of both the clock and the
labelling as this data can give.

**The two devices on the PV frames do not.** `c2g-9FFB8A0D0` (CF2) and
`c2g-9FFB8AA78` (CF4) disagree in a way the shift does not explain — their import
correlation at the winning offset is only **0.22 and 0.21**, against 0.82–0.92 for the
other three. Two symptoms, both on the same stable stretch:

* **mySET exports more.** 8A0D0: 267.7 kWh against MQTT's 174.9 (**1.53x**); 8AA78:
  1395.4 against 1342.8 (1.04x), with the export correlation still high (0.72 and 0.92),
  so the shape matches and the level does not.
* **mySET barely imports at all.** 8A0D0 reads 0.4 kWh of import over the twelve days
  where MQTT reads 3.3; 8AA78 reads 1.0 kWh where MQTT reads 33.2. The MQTT import on
  those devices is not noise — it arrives in real evening runs, exactly when a PV system
  stops covering the house.

That pattern looks like the two meters netting import against export differently at a
PV site, but the data here does not prove it. Which source is right for those two is
**not resolved**: it is an open question carried into the findings and into notebook 02,
which has to decide whether their pre-May history can be trained on.


In [ ]:
# Merge the two provenances into one frame, with a clean cut per device.
#
# The export itself carries no counting frame — it is the DSO's meter, not the MQTT
# gateway — so `cf_type` here is pure provenance, copied from the frame each device
# uses in the MQTT table. Section 2 established that a device uses exactly one frame
# for its whole history, so the copy is unambiguous.
myset_frames = {}
first_mqtt_ts = {}
for device_id, g in df_meters[df_meters["device_id"].isin(MYSET_DEVICES)].groupby("device_id"):
    myset_frames[device_id] = sorted(g["cf_type"].unique())[0]
    first_mqtt_ts[device_id] = g["ts"].min()
print("frame per device (from the MQTT table):", myset_frames)

# `_id` is a bigint on the MQTT side and an md5 text on the mySET side; cast both to
# string so the column survives the concat. Notebook 02 never reads it.
df_meters["_id"] = df_meters["_id"].astype(str)
df_meters["source"] = "mqtt"

# `estimated` is False on every row of the export and `_sdc_extracted_at` is pipeline
# bookkeeping, so neither is carried over.
myset_part = df_myset[["_id", "device_id", "ts", "consumption_kwh", "production_kwh"]].copy()
myset_part["_id"] = myset_part["_id"].astype(str)
myset_part["cf_type"] = myset_part["device_id"].map(myset_frames)
myset_part["meter_type"] = "M1"
myset_part["source"] = "myset"

# CLEAN CUT: mySET strictly before each device's first MQTT reading, MQTT from then on.
# No interleaving inside the overlap, so the untrustworthy tail is never used.
keep = pd.Series(False, index=myset_part.index)
for device_id, cut in first_mqtt_ts.items():
    is_device = myset_part["device_id"] == device_id
    keep = keep | (is_device & (myset_part["ts"] < cut))
myset_part = myset_part[keep]

print("\nrows added per device (mySET, before the first MQTT reading):")
print(myset_part.groupby("device_id").size().to_string())
print("rows of df_myset discarded as inside/after the MQTT window:",
      f"{len(df_myset) - len(myset_part):,}")

rows_before = len(df_meters)
df_meters = pd.concat([df_meters, myset_part[df_meters.columns]], ignore_index=True)
df_meters = df_meters.sort_values(["device_id", "ts"]).reset_index(drop=True)
validate_raw_schema(df_meters, kind="meter")

print(f"\nmeter rows: {rows_before:,} -> {len(df_meters):,} "
      f"(+{len(df_meters) - rows_before:,})")
print("duplicates on (device_id, ts):",
      int(df_meters.duplicated(subset=["device_id", "ts"]).sum()))
print("\nthe five devices after the merge:")
five = df_meters[df_meters["device_id"].isin(MYSET_DEVICES)]
print(five.groupby("device_id").agg(rows=("ts", "size"),
                                    first_ts=("ts", "min"),
                                    last_ts=("ts", "max")).to_string())

# The DST hole of 2025-10-26: the export held one copy of the repeated local hour, so
# the UTC series loses two hours, not one. max_gap_hours is 1 in the default config,
# so notebook 02's regular grid will not interpolate it.
hole_device = MYSET_DEVICES[0]
hole_from = pd.Timestamp("2025-10-25 22:00", tz="UTC")
hole_to = pd.Timestamp("2025-10-26 03:00", tz="UTC")
expected = pd.date_range(hole_from, hole_to, freq="15min")
is_hole_device = df_meters["device_id"] == hole_device
present = set(df_meters.loc[is_hole_device, "ts"])
missing = []
for slot in expected:
    if slot not in present:
        missing.append(str(slot))
print(f"\n2025-10-26 hole check for {hole_device}, {hole_from} -> {hole_to}:")
print(f"{len(missing)} of {len(expected)} quarters missing:", missing)


`df_meters` now carries **two provenances in one frame**, told apart by the new
`source` column: `mqtt` for everything read from `ds_dev_silver.meters_data`, `myset`
for the quarters that only the DSO export has. For the five devices that means a
history reaching back to **January–March 2025** instead of 2026-05-21, on the same UTC
clock, the same kWh-per-quarter units and the same contract columns as the rest — the
package's own `validate_raw_schema` accepts the merged frame. Everything downstream in
this notebook (the timeline, the daily heatmap, the value distributions, the profiles,
the Open-Meteo download window and the processed frame) sees the longer history, and
the parquet extract notebook 02 reads carries the `source` column with it.


In [ ]:
# Per-device span and hourly coverage: how many of the hourly slots between a
# device's first and last reading actually carry data.
dev = df_meters.groupby("device_id").agg(
    readings=("ts", "size"),
    first_ts=("ts", "min"),
    last_ts=("ts", "max"),
    mean_cons=("consumption_kwh", "mean"),
    mean_prod=("production_kwh", "mean"),
)

# The counting frames a device uses, joined into one label ("CF2" or "CF2+CF4").
frame_labels = {}
for device_id, device_rows in df_meters.groupby("device_id"):
    device_frames = sorted(device_rows["cf_type"].unique())
    frame_labels[device_id] = "+".join(device_frames)
dev["frames"] = pd.Series(frame_labels)

dev["span_days"] = (dev["last_ts"] - dev["first_ts"]).dt.total_seconds() / 86400

# Distinct hours that carry at least one reading, per device.
hours = df_meters[["device_id"]].copy()
hours["hour"] = df_meters["ts"].dt.floor("h")
hours_present = hours.groupby("device_id")["hour"].nunique()

dev["hours_present"] = hours_present
dev["hours_expected"] = (dev["span_days"] * 24).round().astype(int) + 1
dev["coverage_pct"] = 100 * dev["hours_present"] / dev["hours_expected"]
dev["has_pv"] = dev["mean_prod"] > 0.01
dev = dev.sort_values("first_ts")

print("devices:", len(dev))
print("with PV export:", int(dev["has_pv"].sum()))
print("consumption-only:", int((~dev["has_pv"]).sum()))

print("\nonboarding cohorts (first reading date, UTC):")
print(dev["first_ts"].dt.date.value_counts().sort_index())

print("\nspan / coverage summary:")
print(dev[["span_days", "coverage_pct", "readings"]].describe().round(1))

dev_view = dev.drop(columns=["hours_expected"])
dev_view.round(3).head(12)

### Device timeline

One bar per device, from its first to its last reading, coloured by whether the meter
ever exports (mean production above the 0.01 kWh/15 min activity floor the pipeline
uses). The onboarding waves are immediately visible.

In [ ]:
# One horizontal bar per device: from its first to its last reading.
order = dev.sort_values("first_ts")
y = np.arange(len(order))

# Matplotlib needs naive dates, so drop the timezone after converting to UTC.
first_ts_utc = order["first_ts"].dt.tz_convert("UTC")
first_ts_naive = first_ts_utc.dt.tz_localize(None)
starts = mdates.date2num(first_ts_naive)
widths = order["span_days"].values

# Device ids are long UUIDs; the last five characters are enough to tell them apart.
device_labels = []
for device_id in order.index:
    device_labels.append(device_id[-5:])

exports = order["has_pv"].values
imports_only = ~exports

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(y[exports], widths[exports], left=starts[exports],
        color="tab:blue", label="with PV export")
ax.barh(y[imports_only], widths[imports_only], left=starts[imports_only],
        color="tab:orange", label="consumption only")
ax.set_yticks(y)
ax.set_yticklabels(device_labels, fontsize=7)
ax.invert_yaxis()
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.set_title("M1 devices — data span (first to last 15-min reading)")
ax.set_xlabel("date (UTC)")
ax.set_ylabel("device (last 5 chars of id)")
ax.legend()
plt.tight_layout()
plt.show()

### Readings per day

A device that is healthy delivers **96 readings/day** (4 per hour). The heatmap shows
per-device daily counts: white is "no data at all", dark blue is a full day, and
anything in between is a partial day.

In [ ]:
# Count the readings of every device on every calendar day (UTC).
per_device_day = df_meters[["device_id"]].copy()
ts_utc = df_meters["ts"].dt.tz_convert("UTC")
per_device_day["day"] = ts_utc.dt.floor("D")
daily_counts = per_device_day.groupby(["device_id", "day"]).size()
daily = daily_counts.rename("n").reset_index()

# One row per device (same order as the timeline above), one column per day.
mat = daily.pivot(index="device_id", columns="day", values="n")
mat = mat.reindex(order.index)
days = mat.columns

device_labels = []
for device_id in mat.index:
    device_labels.append(device_id[-5:])

first_day_num = mdates.date2num(days[0].tz_localize(None))
last_day_num = mdates.date2num(days[-1].tz_localize(None))

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(mat.values, aspect="auto", cmap="Blues", vmin=0, vmax=96,
               interpolation="nearest",
               extent=[first_day_num, last_day_num, len(mat), 0])
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.set_yticks(np.arange(len(mat)) + 0.5)
ax.set_yticklabels(device_labels, fontsize=7)
ax.set_title("Daily reading count per device — white means no data that day")
ax.set_xlabel("date (UTC)")
ax.set_ylabel("device (last 5 chars of id)")
fig.colorbar(im, ax=ax, label="readings per day (96 = complete)")
plt.tight_layout()
plt.show()

# A healthy device delivers 96 readings a day (4 per hour).
is_complete_day = daily["n"] == 96
print(f"device-days with the full 96 readings: {is_complete_day.mean() * 100:.1f}%")
print("distribution of daily reading counts:")
print(daily["n"].describe().round(1))

## 5. Meters — the values themselves

All quantities below are **kWh per 15-minute interval**.

In [ ]:
# Distribution of the raw 15-minute values, with the far tail spelled out.
value_stats = df_meters[["consumption_kwh", "production_kwh"]].describe(
    percentiles=[0.5, 0.9, 0.99, 0.999])
print("per-15-min value distribution (kWh):")
print(value_stats.round(4))

# How much of each series is exactly zero.
is_zero_consumption = df_meters["consumption_kwh"] == 0
is_zero_production = df_meters["production_kwh"] == 0
is_zero_both = is_zero_consumption & is_zero_production
print(f"\nexact zeros: consumption {is_zero_consumption.mean() * 100:.1f}%, "
      f"production {is_zero_production.mean() * 100:.1f}%, "
      f"both {is_zero_both.mean() * 100:.1f}%")

In [ ]:
# One histogram per direction, zeros excluded so the shape of the rest is visible.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col, color, name in [
    (axes[0], "consumption_kwh", "tab:blue", "Grid import"),
    (axes[1], "production_kwh", "tab:orange", "Grid export"),
]:
    is_positive = df_meters[col] > 0
    ax.hist(df_meters.loc[is_positive, col], bins=80, color=color)
    ax.set_yscale("log")
    ax.set_title(f"{name} — non-zero readings")
    ax.set_xlabel("kWh per 15 min")
    ax.set_ylabel("readings (log scale)")
plt.tight_layout()
plt.show()

# The zero readings left out of the two panels.
for col in ["consumption_kwh", "production_kwh"]:
    share = (df_meters[col] == 0).mean() * 100
    print(f"{col}: {share:.0f}% of readings are exactly zero and are not plotted")

In [ ]:
# Share of exactly-zero readings per device, one column per direction.
zero_flags = df_meters[["device_id"]].copy()
zero_flags["cz"] = df_meters["consumption_kwh"] == 0
zero_flags["pz"] = df_meters["production_kwh"] == 0

zero_share = zero_flags.groupby("device_id")[["cz", "pz"]].mean()
zeros = (zero_share * 100).round(1)
zeros = zeros.rename(columns={"cz": "zero_consumption_pct", "pz": "zero_production_pct"})
zeros = zeros.join(dev[["has_pv", "readings"]])
zeros = zeros.sort_values("zero_consumption_pct", ascending=False)
print("share of exact zeros per device (%):")
zeros.head(10)

In [ ]:
# The twelve devices with the highest average import.
topn = dev.nlargest(12, "mean_cons")

device_labels = []
for device_id in topn.index:
    device_labels.append(device_id[-5:])

fig, ax = plt.subplots(figsize=(10, 4))
yy = np.arange(len(topn))
ax.barh(yy, topn["mean_cons"].values, color="tab:blue")
ax.set_yticks(yy)
ax.set_yticklabels(device_labels)
ax.invert_yaxis()
ax.set_title("Highest average grid import — top 12 devices")
ax.set_xlabel("mean consumption (kWh per 15 min)")
ax.set_ylabel("device (last 5 chars of id)")
plt.tight_layout()
plt.show()

### Mean daily profile, local time

Timestamps are UTC; the human rhythm (and the sun) follow **Europe/Rome**, so the
profile is computed on local time. PV and consumption-only devices are separated
because their shapes are opposite.

In [ ]:
# Add the local-time columns the daily profile needs, plus the PV flag per device.
loc = df_meters[["device_id", "ts", "consumption_kwh", "production_kwh"]].copy()
loc["ts_local"] = loc["ts"].dt.tz_convert(LOCAL_TZ)
loc["hour_local"] = loc["ts_local"].dt.hour
loc["dow"] = loc["ts_local"].dt.dayofweek
loc = loc.merge(dev[["has_pv"]], left_on="device_id", right_index=True, how="left")

# Mean value per hour of the local day, PV and consumption-only kept apart.
prof = loc.groupby(["has_pv", "hour_local"])[["consumption_kwh", "production_kwh"]].mean()

n_pv = int(dev["has_pv"].sum())
n_no_pv = int((~dev["has_pv"]).sum())

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True)
for ax, pv, title in [
    (axes[0], True, f"PV devices (n={n_pv})"),
    (axes[1], False, f"Consumption-only devices (n={n_no_pv})"),
]:
    p = prof.loc[pv]
    ax.plot(p.index, p["consumption_kwh"], color="tab:blue", label="import (consumption)")
    ax.plot(p.index, p["production_kwh"], color="tab:orange", label="export (production)")
    ax.set_xticks(range(0, 24, 3))
    ax.set_ylim(bottom=0)
    ax.set_title(title)
    ax.set_xlabel("hour of day (Europe/Rome)")
    ax.set_ylabel("mean energy (kWh per 15 min)")
    ax.legend()
fig.suptitle("Mean daily profile of grid exchange")
plt.tight_layout()
plt.show()

In [ ]:
# Mean value per day of the local week, PV and consumption-only kept apart.
dow_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
wk = loc.groupby(["has_pv", "dow"])[["consumption_kwh", "production_kwh"]].mean()

import_no_pv = wk.loc[False]["consumption_kwh"].reindex(range(7))
import_pv = wk.loc[True]["consumption_kwh"].reindex(range(7))

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(7)
w = 0.38
ax.bar(x - w / 2, import_no_pv.values, width=w, color="tab:orange",
       label="consumption-only devices — import")
ax.bar(x + w / 2, import_pv.values, width=w, color="tab:blue",
       label="PV devices — import")
ax.set_xticks(x)
ax.set_xticklabels(dow_names)
ax.set_title("Weekly seasonality of grid import")
ax.set_xlabel("day of week (Europe/Rome)")
ax.set_ylabel("mean import (kWh per 15 min)")
ax.legend()
plt.tight_layout()
plt.show()

# The same split reduced to weekday vs weekend.
weekend_view = loc[["has_pv", "consumption_kwh", "production_kwh"]].copy()
weekend_view["weekend"] = loc["dow"] >= 5
weekend_means = weekend_view.groupby(["has_pv", "weekend"])[
    ["consumption_kwh", "production_kwh"]].mean()
print("weekday vs weekend mean (kWh/15min):")
print(weekend_means.round(4))

### Two example weeks

One mature device with PV and one consumption-only device, same calendar week, raw
15-minute resolution — the shape the models have to reproduce.

In [ ]:
# Pick the biggest exporter among the devices that were already online in Sep 2025,
# and the biggest importer among the consumption-only ones.
is_mature = dev["first_ts"] < pd.Timestamp("2025-10-01", tz="UTC")
mature = dev[is_mature]
mature_with_pv = mature[mature["has_pv"]]
pv_dev = mature_with_pv["mean_prod"].idxmax()
nopv_pool = dev[~dev["has_pv"]]
nopv_dev = nopv_pool["mean_cons"].idxmax()

# Take a full week three weeks after both of them are online.
common_start = max(dev.loc[pv_dev, "first_ts"], dev.loc[nopv_dev, "first_ts"])
week_start = (common_start + pd.Timedelta(days=21)).floor("D")
week_end = week_start + pd.Timedelta(days=7)

print(f"example week: {week_start:%Y-%m-%d} -> {week_end:%Y-%m-%d} (UTC)")
print(f"PV device: {pv_dev} (mean export {dev.loc[pv_dev, 'mean_prod']:.3f} kWh/15min)")
print(f"consumption-only: {nopv_dev} (mean import {dev.loc[nopv_dev, 'mean_cons']:.3f} kWh/15min)")

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
for ax, device, title in [
    (axes[0], pv_dev, "Mature device with PV — import and export"),
    (axes[1], nopv_dev, "Consumption-only device — import"),
]:
    is_device = df_meters["device_id"] == device
    in_week = (df_meters["ts"] >= week_start) & (df_meters["ts"] < week_end)
    s = df_meters[is_device & in_week]
    s = s.sort_values("ts")
    tl = s["ts"].dt.tz_convert(LOCAL_TZ)
    ax.plot(tl, s["consumption_kwh"], color="tab:blue", label="import")
    if device == pv_dev:
        ax.plot(tl, s["production_kwh"], color="tab:orange", label="export")
    ax.set_ylim(bottom=0)
    ax.set_title(title)
    ax.set_ylabel("kWh per 15 min")
    ax.legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%a %d"))
axes[1].set_xlabel("local time (Europe/Rome)")
plt.tight_layout()
plt.show()

## 6. Weather from the gold layer

The upstream pipeline already materialises hourly weather **features** for the meter
site. Loaded, again, through the package loader so it matches the CLI.

Its `datetime` column is a **naive timestamp in local (Europe/Rome) time** — the
package's `prepare_weather` localises naive timestamps with `config.local_tz`, which
is consistent. We verify it below rather than trust it.

In [ ]:
# Same package loader as the meters, one source table and no filters.
df_weather_db = load_weather_from_db([{"table": WEATHER_TABLE}], engine=engine)

print("shape:", df_weather_db.shape)
print("datetime dtype:", df_weather_db["datetime"].dtype, "(naive = local time)")
print("span:", df_weather_db["datetime"].min(), "->", df_weather_db["datetime"].max())
print("extracted at:", df_weather_db["_sdc_extracted_at"].min(),
      "->", df_weather_db["_sdc_extracted_at"].max())
df_weather_db.head(3)

In [ ]:
# Everything except the timestamp and the extraction stamp is a weather feature.
feature_cols = []
for column in df_weather_db.columns:
    if column in ("datetime", "_sdc_extracted_at"):
        continue
    feature_cols.append(column)

feature_stats = df_weather_db[feature_cols].describe().T
feature_stats.round(3)

In [ ]:
# Sort by time, then look for jumps of more than one hour in the hourly series.
w = df_weather_db.sort_values("datetime").reset_index(drop=True)
step = w["datetime"].diff()
is_break = step > pd.Timedelta("1h")
breaks = w.loc[is_break]

# Each break closes the island that was open and starts the next one.
islands = []
start = w["datetime"].iloc[0]
for _, row in breaks.iterrows():
    before_break = w["datetime"] < row["datetime"]
    prev = w["datetime"][before_break].max()
    islands.append((start, prev))
    start = row["datetime"]
islands.append((start, w["datetime"].iloc[-1]))

isl = pd.DataFrame(islands, columns=["from", "to"])
island_hours = (isl["to"] - isl["from"]).dt.total_seconds() / 3600 + 1
isl["hours"] = island_hours.astype(int)

# How many rows each island actually carries, against the hours it spans.
rows_per_island = []
for island_start, island_end in islands:
    inside_island = (w["datetime"] >= island_start) & (w["datetime"] <= island_end)
    rows_per_island.append(int(inside_island.sum()))
isl["rows"] = rows_per_island
isl["complete_pct"] = (100 * isl["rows"] / isl["hours"]).round(1)
print("coverage islands (naive local time):")
print(isl)

# The holes between consecutive islands (plain lists, so the rows line up).
gap_starts = isl["to"][:-1].tolist()
gap_ends = isl["from"][1:].tolist()
gaps = pd.DataFrame({"gap_starts_after": gap_starts, "gap_ends_at": gap_ends})
gap_hours = (gaps["gap_ends_at"] - gaps["gap_starts_after"]).dt.total_seconds() / 3600 - 1
gaps["missing_hours"] = gap_hours.astype(int)
gaps["missing_days"] = (gaps["missing_hours"] / 24).round(1)
print("\ngaps:")
print(gaps)

In [ ]:
# Rows per calendar day, reindexed over the whole meter span so holes show as zero.
per_day_source = w[["datetime"]].copy()
per_day_source["day"] = w["datetime"].dt.floor("D")
per_day = per_day_source.groupby("day").size()

meters_start_local = df_meters["ts"].min().tz_convert(LOCAL_TZ)
first_day = meters_start_local.floor("D").tz_localize(None)
last_day = w["datetime"].max().floor("D")
full_days = pd.date_range(first_day, last_day, freq="D")
per_day = per_day.reindex(full_days, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.fill_between(per_day.index, per_day.values, step="mid", color="tab:blue")
ax.axhline(24, color="grey", linestyle="--", label="24 h = complete day")

# Shade every gap found above, and the stretch before the weather table starts.
for _, g in gaps.iterrows():
    ax.axvspan(g["gap_starts_after"], g["gap_ends_at"], color="tab:orange", alpha=0.3)
ax.axvspan(per_day.index[0], w["datetime"].min(), color="grey", alpha=0.2)

ax.set_ylim(0, 30)
ax.set_title("Gold-layer weather coverage — hours available per day")
ax.set_xlabel("date (local time)")
ax.set_ylabel("rows per day (h)")
ax.legend()
plt.tight_layout()
plt.show()

print("the meters start", df_meters["ts"].min().date(),
      "but the weather table only", w["datetime"].min().date())
for _, g in gaps.iterrows():
    print("no weather from", g["gap_starts_after"], "to", g["gap_ends_at"],
          f"({g['missing_days']:.0f} days)")

### Where the hole comes from

The hole exists **only in the gold table**. `ds_dev_silver.om_weather_hourly` and
`raw.om_weather_features_meters` both hold all 5,175 hours from 2026-02-20 17:00 to
2026-09-24 07:00 with no gap at all, and raw carries daily `_sdc_extracted_at` stamps
from 2026-04-14 through 2026-09-22 — so no pipeline run was missed. What is absent
from `ds_dev_gold.om_weather_features_meters` is 2026-04-01 00:00 → 2026-07-31 23:00
(2,928 h), cut exactly on calendar-month boundaries and splitting single daily batches
in two: the batch stamped 2026-04-14 reaches 2026-04-14 18:00 in raw but stops at
2026-03-31 23:00 in gold, and the batch stamped 2026-08-01 survives in gold only from
2026-08-01 00:00. That is the signature of a **datetime-range delete on the gold
table**, not of a gap in the source. Who ran it is unknown.

The gold dbt model is incremental on `_sdc_extracted_at > max(gold)`, so it cannot
re-import those rows on its own; a `dbt run --full-refresh` of that model — or a
one-off insert from raw — restores them.

### Timezone check

True solar noon at this longitude is around **11:15 UTC**. If `datetime` were UTC the
mean irradiance curve would peak in the 11:00–12:00 bin in both islands. The two
islands are plotted separately because one is late winter (CET, UTC+1) and one is
summer (CEST, UTC+2): if the column is a **DST-aware local clock**, the peak bin must
*move by one hour* between them.

In [ ]:
# Label every row with the island it belongs to.
second_island_start = islands[1][0]
is_first_island = w["datetime"] < second_island_start
w["island"] = np.where(is_first_island, "island 1 (Feb–Mar)", "island 2 (Aug–Sep)")

# Mean radiation per hour of the stored timestamp, one column per island.
hour_of_day = w["datetime"].dt.hour
mean_radiation = w.groupby(["island", hour_of_day])["shortwave_radiation"].mean()
by_hour = mean_radiation.unstack(0)

fig, ax = plt.subplots(figsize=(10, 4))
for name, color in zip(by_hour.columns, ["tab:blue", "tab:orange"]):
    s = by_hour[name]
    ax.plot(s.index, s.values, color=color, label=name)
    # Mark the peak hour: that is what moves between the two islands.
    peak = int(s.idxmax())
    ax.plot(peak, s.max(), "o", color=color)
    ax.annotate(f"peak {peak:02d}:00", (peak, s.max()),
                textcoords="offset points", xytext=(5, 5))
ax.set_xticks(range(0, 24, 2))
ax.set_title("Mean shortwave radiation by naive hour of the `datetime` column")
ax.set_xlabel("hour of the stored timestamp")
ax.set_ylabel("shortwave radiation (W/m²)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Which reading of `datetime` reproduces the stored `solar_elevation`? Fit the
# site geometry under four hypotheses and compare.
from celine.forecasting.core.weather import solar_position

fits = []

# Three fixed-offset hypotheses: shift the naive stamps back, then call them UTC.
for label, shift in [("naive = UTC", 0), ("naive = fixed UTC+1 (no DST)", 1),
                     ("naive = fixed UTC+2 (no DST)", 2)]:
    shifted = df_weather_db["datetime"] - pd.Timedelta(hours=shift)
    ts = shifted.dt.tz_localize("UTC")
    el, _ = solar_position(pd.DatetimeIndex(ts), SITE_LAT, SITE_LON)
    err = float(np.abs(el - df_weather_db["solar_elevation"]).mean())
    fits.append({"hypothesis": label, "mean_abs_error_deg": round(err, 2)})

# Fourth hypothesis: a DST-aware Europe/Rome clock (the ambiguous hour becomes NaT).
ts_rome_local = df_weather_db["datetime"].dt.tz_localize(LOCAL_TZ, ambiguous="NaT",
                                                         nonexistent="shift_forward")
ts_rome = ts_rome_local.dt.tz_convert("UTC")
ok = ts_rome.notna()
el, _ = solar_position(pd.DatetimeIndex(ts_rome[ok]), SITE_LAT, SITE_LON)
err = float(np.abs(el - df_weather_db["solar_elevation"][ok]).mean())
fits.append({"hypothesis": "naive = Europe/Rome (DST aware)", "mean_abs_error_deg": round(err, 2)})
print("stored `solar_elevation` vs the site's true solar geometry:")
print(pd.DataFrame(fits))

# Where the two column families peak inside each island — they should agree, and do not.
peak_rows = []
for island_name, island_rows in w.groupby("island"):
    island_hour = island_rows["datetime"].dt.hour
    mean_elevation_by_hour = island_rows.groupby(island_hour)["solar_elevation"].mean()
    mean_radiation_by_hour = island_rows.groupby(island_hour)["shortwave_radiation"].mean()
    peak_rows.append({
        "island": island_name,
        "solar_elevation_peak_h": mean_elevation_by_hour.idxmax(),
        "shortwave_peak_h": mean_radiation_by_hour.idxmax(),
    })
peaks = pd.DataFrame(peak_rows).set_index("island")
print("\nhour of the day with the highest mean value, per island:")
print(peaks)

The radiation columns follow a DST-aware local clock, but the stored `solar_elevation`
does **not**: it is best reproduced by reading the timestamps as a *fixed* UTC+1, and
its peak bin stays at 12:00 in both islands instead of moving with DST. The two column
families in the same table therefore disagree by about an hour in summer. An upstream
caveat to report, not something to patch here.

**Sourcing decision.** The gold table holds only ~2.2k hourly rows across two islands,
starts five months *after* the first meter readings, and has a four-month hole in the
middle. It cannot cover the mature meters' history, so from here on the weather used
for analysis is downloaded directly from **Open-Meteo** for the meters' site, and the
gold table is kept as a cross-check and saved alongside.

The differences to keep in mind, since the two sources are *not* interchangeable
column by column, are partly **how the weather was asked for** and partly **what the
columns mean**:

* **elevation** — the tap sends no `elevation`, so Open-Meteo resolves the site on
  its own 90 m DEM and gets 1172 m. Forcing `elevation=1100` on the download makes
  Open-Meteo downscale temperature by lapse rate over the 72 m difference, about
  0.47 °C, which is exactly the constant 0.45 °C offset (std 0.06) this notebook used
  to show against the gold table in August–September. The complement therefore leaves
  the elevation alone, like the pipeline.
* **model** — the pipeline extracts with `models=icon_d2` (DWD ICON-D2, 2.2 km). A
  download that falls back to the ERA5 archive (~31 km) for anything older than 92
  days disagrees there by construction: 2.0 °C mean absolute difference on
  temperature and r = 0.56 on cloud cover over February–March. Asking Open-Meteo for
  `icon_d2` removes that difference.
* **`global_tilted_irradiance`** — the tap requests no tilt or azimuth, so the gold
  column labelled *tilted* is in fact plain horizontal irradiance: it equals
  `shortwave_radiation` in 96 % of rows (max difference 1.8 W/m²). The package
  requests tilt 30°, azimuth 0 (south), and reads about 19 % higher during daylight.
* **`effective_solar_pv`** — the gold table stores `direct + 0.9 · diffuse` in W/m²
  (hundreds), while the package computes `cos(zenith)` clipped to `[0, 1]`.
* **`cloud_cover_diff`** — the gold table stores the *absolute* hourly change, the
  package a *signed* first difference.
* **`solar_elevation`** — stored unclipped (it goes down to about −55° at night) whereas
  the package clips it at 0; and, as shown above, it is about an hour off during
  summer time.
* **`theoretical_prod`** — the gold table multiplies tilted irradiance by its own
  W/m²-scale `effective_solar_pv`, which yields values in the 10⁵ range; the package
  recomputes the same name as irradiance × cos(zenith), i.e. W/m². `build_processed_hourly`
  silently **overwrites** the stored column, so the mismatch does not propagate — but
  anything reading the gold table directly would be comparing two different quantities.
* **`is_daylight`** — flagged 1 for about 62 % of all rows, and it disagrees with
  `solar_elevation > 0` on 11 % of them, so it cannot be taken at face value.

## 7. Weather from Open-Meteo — covering the full meter span

`download_weather_features` splits the window automatically. With `model="icon_d2"`
the history comes from Open-Meteo's **Historical Forecast API**
(`historical-forecast-api.open-meteo.com`), which serves ICON-D2 back to 2022 and up
to today with no nulls, while the next 48 h come from the **forecast API** with the
same model; where the two overlap, **history wins**. It returns the 12 contract
features on a **tz-aware UTC** `datetime`.

In [ ]:
# Cover the whole meter history plus two days of forecast.
om_start = df_meters["ts"].min().floor("D")
om_end = pd.Timestamp.now(tz="UTC").floor("D") + pd.Timedelta(days=2)

from celine.forecasting.core.weather import download_weather_features

t0 = time.time()
weather = download_weather_features(
    latitude=SITE_LAT, longitude=SITE_LON,
    start=om_start, end=om_end, model=WEATHER_MODEL,
)
print(f"downloaded in {time.time() - t0:.1f}s — shape {weather.shape}")

first_hour = weather["datetime"].min()
last_hour = weather["datetime"].max()
print("span:", first_hour, "->", last_hour, "(tz-aware UTC)")

# Completeness: how many of the hourly slots in that span carry a row.
expected_hours = int((last_hour - first_hour).total_seconds() / 3600) + 1
print(f"hourly completeness: {len(weather)}/{expected_hours} = "
      f"{100 * len(weather) / expected_hours:.2f}%")
hourly_steps = weather["datetime"].sort_values().diff()
print("gaps > 1h:", int((hourly_steps > pd.Timedelta("1h")).sum()))

# Which columns have missing values at all.
null_counts = weather.isna().sum()
has_nulls = null_counts > 0
nulls = null_counts[has_nulls]
if len(nulls):
    print("columns with nulls:")
    print(nulls)
    # Check the holes form one block rather than being scattered, using whichever
    # column is worst affected.
    worst_col = nulls.idxmax()
    worst_missing = weather[worst_col].isna()
    miss = weather[worst_missing]
    print(f"\n{worst_col} is missing from",
          miss["datetime"].min(), "->", miss["datetime"].max(),
          f"({len(miss)} h = {len(miss) / 24:.0f} days)")
    same_hours = []
    for col in nulls.index:
        if weather[col].isna().equals(worst_missing):
            same_hours.append(col)
    print("columns missing on exactly those hours:", same_hours)
else:
    print("no nulls in any column")
weather.head(3)

In [ ]:
# Meter span vs each weather source.
meter_first_hour = df_meters["ts"].min().floor("h")
meter_last_hour = df_meters["ts"].max().ceil("h")
meter_hours = pd.date_range(meter_first_hour, meter_last_hour, freq="h")

# The gold table's naive local stamps, read as Europe/Rome and moved onto UTC.
db_local = df_weather_db["datetime"].dt.tz_localize(LOCAL_TZ, ambiguous="NaT",
                                                    nonexistent="shift_forward")
db_utc = db_local.dt.tz_convert("UTC")

# Share of the meter hours each source can serve.
db_hours_covered = db_utc.dropna().isin(meter_hours).sum()
om_hours_covered = weather["datetime"].isin(meter_hours).sum()

cov = pd.DataFrame({
    "source": ["gold table", "Open-Meteo"],
    "rows": [len(df_weather_db), len(weather)],
    "covers_meter_hours_pct": [
        round(100 * db_hours_covered / len(meter_hours), 1),
        round(100 * om_hours_covered / len(meter_hours), 1),
    ],
})
print(f"meter span: {len(meter_hours):,} hourly slots "
      f"({df_meters['ts'].min():%Y-%m-%d} -> {df_meters['ts'].max():%Y-%m-%d} UTC)")
print(cov)

### Do the two sources agree?

The gold table's naive local timestamps are converted to UTC before the comparison
(`tz_localize("Europe/Rome", ambiguous="NaT", nonexistent="shift_forward")`, exactly what
`prepare_weather` does). A week inside the second island:

In [ ]:
# Gold-table rows on a UTC clock; the ambiguous DST hour became NaT and is dropped.
db_cmp = df_weather_db.copy()
db_cmp["datetime_utc"] = db_utc
db_cmp = db_cmp.dropna(subset=["datetime_utc"])

# A week inside the second island — fall back to the last full week if it is not covered.
cmp_start = pd.Timestamp("2026-08-05", tz="UTC")
covers_default_week = db_cmp["datetime_utc"].min() <= cmp_start <= db_cmp["datetime_utc"].max()
if not covers_default_week:
    cmp_start = db_cmp["datetime_utc"].max().floor("D") - pd.Timedelta(days=8)
cmp_end = cmp_start + pd.Timedelta(days=7)
print(f"comparison week (UTC): {cmp_start:%Y-%m-%d} -> {cmp_end:%Y-%m-%d}")

in_week_db = (db_cmp["datetime_utc"] >= cmp_start) & (db_cmp["datetime_utc"] < cmp_end)
a = db_cmp[in_week_db].set_index("datetime_utc")
a = a.sort_index()
in_week_om = (weather["datetime"] >= cmp_start) & (weather["datetime"] < cmp_end)
b = weather[in_week_om].set_index("datetime")
b = b.sort_index()

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
for ax, col, unit in [(axes[0], "shortwave_radiation", "W/m²"),
                      (axes[1], "temperature_2m", "°C")]:
    ax.plot(a.index, a[col], color="tab:blue", label="gold table")
    ax.plot(b.index, b[col], color="tab:orange", linestyle="--", label="Open-Meteo")
    ax.set_title(col.replace("_", " "))
    ax.set_ylabel(f"{col} ({unit})")
    ax.legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%a %d"))
axes[1].set_xlabel("time (UTC)")
fig.suptitle("Gold table vs Open-Meteo — same week, same site")
plt.tight_layout()
plt.show()

# Numeric agreement on the hours both sources have.
joined = a.join(b, how="inner", lsuffix="_db", rsuffix="_om")
rows = []
for col in ["shortwave_radiation", "temperature_2m", "cloud_cover",
            "global_tilted_irradiance", "effective_solar_pv", "cloud_cover_diff"]:
    x = joined[f"{col}_db"]
    y = joined[f"{col}_om"]
    rows.append({"feature": col,
                 "corr": round(x.corr(y), 3),
                 "mean_db": round(x.mean(), 2),
                 "mean_om": round(y.mean(), 2),
                 "mean_abs_diff": round((x - y).abs().mean(), 2)})
print("\nagreement on the overlapping hours of that week:")
print(pd.DataFrame(rows))

The Open-Meteo complement is gap-free: `download_weather_features` takes the ICON-D2
history from the Historical Forecast API, which has no unfinalised tail. An earlier
extract did carry an 18-day hole (2026-06-22 → 2026-07-08 21:00), but that was a
**package bug, not a property of the source**: `download_raw_weather` let the forecast
frame override the archive frame over their 92-day overlap, and the forecast API
returns all-null rows for the oldest ~17 days of a 92-day `past_days` window, while the
archive had full data there. Worse, `build_weather_features` then **zero-filled**
radiation and cloud cover inside that window, so the old extract carried 17 days of
fake zero irradiance and zero cloud cover, not merely missing temperature. History now
wins on overlap and all-null rows are dropped instead of zero-filled.

Radiation, temperature and cloud cover line up. `global_tilted_irradiance` does not,
and is not supposed to: the tap requests no tilt or azimuth, so the gold column is
horizontal irradiance under a tilted name, while the package asks for tilt 30° /
azimuth 0. `effective_solar_pv` and `cloud_cover_diff` do **not** line up either, and
are expected not to — they are different quantities under the same name (see the note
in section 6). Mixing the two sources in one feature frame would therefore be a silent
bug; the pipeline uses one source at a time, and here that source is Open-Meteo.

## 8. A first joint look

One pass of `build_processed_hourly` — the package's full cleaning chain (hourly
aggregation by SUM, derived `grid_import` / `grid_export` / `net_exchange` with a
noise floor, a regular hourly grid with gap flags, weather merge, calendar features,
outlier flags). The heavy walkthrough of that chain belongs to notebook 02; here it
is only the input the EDA helpers need.

In [ ]:
from celine.forecasting.core.cleaning import build_processed_hourly
from celine.forecasting.core.config import load_config
from celine.forecasting.core.eda import hourly_profiles, segment_devices, weather_correlations

# One pass of the package's cleaning chain, with the Open-Meteo weather.
config = load_config()
processed = build_processed_hourly(df_meters, config, df_weather=weather)
print("shape:", processed.shape)

gap_share_pct = processed["gap_flag"].mean() * 100
print(f"gap_flag rows: {gap_share_pct:.1f}% "
      f"({int(processed['gap_flag'].sum()):,} of {len(processed):,})")
print(f"weather coverage on the grid: {processed['temperature_2m'].notna().mean() * 100:.1f}%")
print("hourly grid:", processed["ts_hour"].min(), "->", processed["ts_hour"].max())
print(f"{processed['ts_hour'].nunique():,} hours x {processed['device_id'].nunique()} devices")

Note what the regular grid does: **every** device is reindexed onto the *global* hourly
range, so a device onboarded in June 2026 carries ten months of `gap_flag = True` rows.
That is why the gap share is so large — it is an artefact of the shared grid, not of
missing readings, and it is the main thing notebook 02 has to decide how to treat.

In [ ]:
# Net importer / net exporter / balanced, decided on the non-gap hours.
segments = segment_devices(processed)
print("device segmentation (mean kWh/h over non-gap hours):")
print(segments["segment"].value_counts())

# Show the same table with short device labels instead of the full ids.
segments_view = segments.copy()
segments_view["device"] = segments["device_id"].str[-5:]
segments_view = segments_view.drop(columns="device_id")
segments_view = segments_view.sort_values("mean_net", ascending=False)
segments_view.head(12)

In [ ]:
# Fleet mean profile by hour of the local day, gap hours excluded by the helper.
prof_h = hourly_profiles(processed)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(prof_h.index, prof_h["grid_import"], color="tab:blue", label="grid import")
ax.plot(prof_h.index, prof_h["grid_export"], color="tab:orange", label="grid export")
ax.plot(prof_h.index, prof_h["net_exchange"], color="tab:green",
        label="net exchange (export − import)")
ax.axhline(0, color="grey")
ax.set_xticks(range(0, 24, 2))
ax.set_title("Fleet mean hourly profile — all devices, gap hours excluded")
ax.set_xlabel("hour of day (Europe/Rome)")
ax.set_ylabel("energy (kWh per hour)")
ax.legend()
plt.tight_layout()
plt.show()
prof_h.head()

In [ ]:
# Weather correlations on the exporting devices only.
exports_energy = segments["mean_export"] > 0.01
pv_devices = segments.loc[exports_energy, "device_id"].tolist()
is_pv_device = processed["device_id"].isin(pv_devices)
corr = weather_correlations(processed[is_pv_device], config)
# `is_daylight` is constant inside the daylight-only subset the helper selects, so its
# correlation is undefined — drop the empty column rather than plot a blank stripe.
corr = corr.dropna(axis=1, how="all")

fig, ax = plt.subplots(figsize=(10, 3))
sns.heatmap(corr, ax=ax, cmap="coolwarm", center=0, vmin=-1, vmax=1,
            annot=True, fmt=".2f", cbar_kws={"label": "Pearson r"})
ax.set_title(f"Weather correlations, daylight hours only — {len(pv_devices)} PV devices")
plt.tight_layout()
plt.show()

In [ ]:
# One mature PV device, real (non-gap) daylight hours with an irradiance value.
is_device = processed["device_id"] == pv_dev
is_real_hour = ~processed["gap_flag"]
is_daylight_hour = processed["is_daylight"] == 1
sub = processed[is_device & is_real_hour & is_daylight_hour]
sub = sub.dropna(subset=["global_tilted_irradiance"])

# Median export per irradiance bin, to show the trend through the cloud of points.
bins = pd.cut(sub["global_tilted_irradiance"], bins=20)
med = sub.groupby(bins, observed=True).agg(x=("global_tilted_irradiance", "median"),
                                           y=("grid_export", "median"))

r = sub["global_tilted_irradiance"].corr(sub["grid_export"])

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(sub["global_tilted_irradiance"], sub["grid_export"], s=9, alpha=0.3,
           color="tab:blue")
ax.plot(med["x"], med["y"], color="tab:orange", label="binned median")
ax.set_title(f"Grid export vs tilted irradiance — one mature PV device (r = {r:.2f})")
ax.set_xlabel("global tilted irradiance (W/m²)")
ax.set_ylabel("grid export (kWh per hour)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"{len(sub):,} daylight hours plotted")

## 9. Save the extracts

Three parquet files, written to `notebooks/data/` (ignored by git). Notebook 02 reads
these instead of hitting the database again.

In [ ]:
# Notebook 02 reads these three files instead of querying the database again.
raw_meters_path = DATA_DIR / "raw_meters_m1.parquet"
weather_db_path = DATA_DIR / "weather_db.parquet"
weather_om_path = DATA_DIR / "weather_openmeteo.parquet"

df_meters.to_parquet(raw_meters_path, index=False)
df_weather_db.to_parquet(weather_db_path, index=False)
weather.to_parquet(weather_om_path, index=False)

summary_rows = []
for path, frame in [(raw_meters_path, df_meters),
                    (weather_db_path, df_weather_db),
                    (weather_om_path, weather)]:
    summary_rows.append({"file": path.name,
                         "rows": len(frame),
                         "cols": frame.shape[1],
                         "size_MB": round(path.stat().st_size / 1e6, 2)})
# The meters frame now carries two provenances, so count the rows of each.
print("meter rows per source:")
print(df_meters["source"].value_counts())
print()
print("written to", DATA_DIR)
pd.DataFrame(summary_rows)

## 10. Findings

Written after looking at the outputs above. All figures are the ones this run produced.

### What the data looks like

1. **41 M1 devices, 834,903 readings, 2024-12-31 23:15 → 2026-09-22 15:15 UTC
   (629.7 days).** The frame now carries **two provenances**, told apart by the
   `source` column: **603,510** readings from the MQTT silver table
   `ds_dev_silver.meters_data` and **231,393** from the mySET export of the DSO's meter
   (section 4). The MQTT table is live — it gains a few rows per run — and the loader
   discards exactly 36 of them as duplicate `(device_id, ts)` (see finding 7). Each
   device uses **exactly one** counting frame for its whole history (in the MQTT table
   CF4 235,203 rows, CF2 162,338, CF3 105,317, CF1 100,688), so the frame is a device
   property, not a per-reading one.

2. **The data is clean at the value level, on both sources.** MQTT side: no nulls, no
   negatives, every timestamp exactly on the 15-minute grid (zero seconds, four minute
   buckets of 150,804 / 150,728 / 150,937 / 151,041 readings), max 6.998 kWh/15 min
   imported and 4.781 exported. mySET side: no nulls, no negatives, `estimated` false
   on every one of its 240,208 rows, and the four minute buckets exactly equal at
   60,052 rows each. In the merged frame the maxima are 6.998 kWh imported and 4.979
   exported. Whatever work notebook 02 has to do, it is about *coverage and alignment*,
   not about bad numbers.

3. **The cohort is still dominated by late joiners, but the mature group is now 15
   devices instead of 10.** Fifteen devices predate 2026-05: the five mySET devices
   (three from 2024-12-31 23:15 UTC — 2025-01-01 00:15 local — one from 2025-02-24 and
   one from 2025-03-04), the eight of the 2025-08-26 wave, one from 2025-09-15 and one
   from 2026-02-03. The other **26** arrive from 2026-05-21 on (12 on 2026-05-21, 10 on
   2026-06-11, 1 on 2026-06-17, 2 on 2026-06-22, 1 on 2026-06-26) and none of them can
   have more than 124 days. Median span is **124.0 days** against a maximum of **629.7**
   and a mean of 226.0; the shortest is 12.3 days. Hourly coverage within each device's
   own span is good (median **97.4 %**, min 46.4 %), and **70.8 %** of device-days carry
   the full 96 readings.

4. **17 of 41 devices export**, 24 are consumption-only. The two groups behave
   oppositely: PV devices average 0.045 kWh/15 min of import against 0.486 for the
   consumption-only ones, and over non-gap hours the segmentation splits 20 net
   importers / 15 net exporters / 6 balanced.

5. **The series are heavily zero-inflated.** 27.8 % of consumption readings and 78.5 %
   of production readings are *exactly* zero, 8.7 % are both. One device
   (`c2g-9FFB88EEC`) is 86.7 % zero on import and 100 % zero on export. The default
   config already answers part of this with a Tweedie objective for `grid_import`; the
   export side deserves the same scrutiny.

6. **Seasonality is where it should be.** Daily profiles show the expected midday
   export bump for PV devices and a flat-plus-evening import shape for the others; the
   weekly effect is now essentially absent (0.4852 weekday vs 0.4877 weekend kWh/15 min
   for consumption-only devices — the weekend is marginally *higher*), so calendar
   features will do far less work than weather and lags.

### Two data artefacts worth naming

7. **The 36 duplicates are a DST artefact, not a frame conflict.** They are four
   quarter-hours on **2025-10-26**, one per each of the nine devices online at the
   time, all within a single frame. That is the night the local clock repeats
   02:00–02:59; the upstream dedup key uses local time, so the repeated wall-clock
   hour survives as two UTC instants. The loader keeps the first and drops 36
   readings.

8. **16 rows are labelled with the wrong measurement point, and each one costs two
   readings.** One `M1` row carries the M2 frame `CF101`, and fifteen `M2` rows carry
   `CF2`/`CF4`. The silver model copies `cf_type` from the MQTT payload's `Type` and
   `meter_type` from its `Meter`, both verbatim, so the inconsistent pair comes
   straight off the device/gateway: the values in those fifteen rows are genuine **M1**
   grid-import readings — 2026-06-10 12:30 reads 1.809 kWh, between M1 neighbours of
   2.009 and 1.880. The `meter_type` + `cf_type` filter removes them, but the damage is
   done earlier: the silver dedup partitions by `(device_id, ts, meter_type)`, so the
   mislabelled message wins the `M2` partition and **evicts the genuine `CF101` PV
   reading** for that quarter. At each of those instants both the M1 row and the true
   M2 row are gone. Rate in the retained raw window: 2 of about 13,000 `CF2` messages.

### The mySET history

9. **Five devices gain up to seventeen months of past that exists nowhere else in the
   database.** `ds_dev_silver.silver_myset_quarterly` holds **240,208** quarter-hour
   readings of the DSO's own meter at the POD for exactly five `sensor_reference`
   values, and they are exactly the five devices whose MQTT history starts on
   2026-05-21: `c2g-9FFB89CF4`, `c2g-9FFB89ED4`, `c2g-9FFB8A0D0`, `c2g-9FFB8AA78`,
   `c2g-DD6C2E6EC`. Three start at 2025-01-01 00:00 local, `c2g-9FFB89ED4` at
   2025-02-24 10:00 and `c2g-9FFB8AA78` at 2025-03-05 00:00; all stop at
   2026-06-08 23:45. The merge keeps everything strictly *before* each device's first
   MQTT reading — **48,533 / 43,309 / 48,533 / 42,485 / 48,533** rows — so the frame
   goes from 603,510 to **834,903** rows and those five devices go from ~124 days to
   **629.7, 575.3, 629.7, 566.7 and 629.7 days** of history. The saved extract is 14.5 MB
   for 834,903 rows and 8 columns.

10. **The naive `ts` is the local wall clock, and it labels a quarter by its start.**
    Neither fact is recorded anywhere upstream; both are recovered from the data. The
    daily row counts settle the clock: 96 rows on every day except **92** on 2025-03-30
    and 2026-03-29 (the spring-forward nights, the only steps larger than 15 minutes in
    the table) and **96** on 2025-10-26, where a local clock has 100 quarters and the
    silver dedup key `(sensor_reference, ts)` has collapsed the repeated hour. The
    alignment scan over the 2026-05-21 → 2026-06-08 overlap settles the labelling:
    at **+15 min** the two sources reach corr 0.922 on import and 72.5 % of quarters
    agreeing to within 1.5 Wh, against 0.880 / 34.3–34.4 % at both 0 and +30 and 0.858
    / 32 % at −15 and +45 — a clean symmetric peak. Read as UTC instead, the series
    needs a −2 h correction to reach the same 0.922, i.e. exactly the CEST offset, which
    is the local reading again (and would be an hour wrong all winter). So:
    `normalize_meters(..., assume_tz="Europe/Rome")`, then **+15 minutes applied in the
    notebook**, because `datasets.yaml` has an `assume_tz` but no `ts_offset`. The cost
    of the DST handling is **20 rows** dropped as `NaT` (the 02:00–02:45 local quarters
    of 2025-10-26, four per device) and, because the export held only one copy of the
    repeated hour, a **two-hour hole** in the UTC series: 8 of the 21 quarters between
    2025-10-25 22:00 and 2025-10-26 03:00 UTC are missing, 00:15 → 02:00 UTC.

11. **The two meters agree quarter by quarter — except on the PV frames, and except at
    the very end of the export.** On the stable part of the overlap (2026-05-22 →
    2026-06-03) the three consumption-only devices match on **100 %, 99.9 % and 100 %**
    of quarters and their twelve-day import totals are identical to the printed decimal
    (862.0, 131.8 and 66.2 kWh on both sides) — two independent meters at the same POD,
    agreeing to the Wh. The two devices on PV frames do not: `c2g-9FFB8A0D0` (CF2) and
    `c2g-9FFB8AA78` (CF4) have an import correlation of only **0.22 and 0.21** over the
    overlap, mySET exports **1.53x** and 1.04x what MQTT reports (267.7 vs 174.9 kWh,
    1395.4 vs 1342.8 kWh), and mySET's import is near zero (0.4 and 1.0 kWh over twelve
    days) where MQTT reads 3.3 and 33.2 kWh in real evening runs. On top of that, the
    **last five days of the export (2026-06-04 → 06-08) are simply wrong**: daily import
    totals run at 1.7–2.0x the MQTT ones and the exact-match share falls from ~0.89 to
    0.21–0.31. The clean-cut merge never uses any of the overlap, so the bad tail cannot
    leak into the extract — but the PV-frame disagreement applies to the *pre-May*
    history too, where there is no second source to check it against.

### Weather

12. **The gold-layer table cannot serve as the history source — and its hole is a
    deletion, not a missed run.** 2,247 hourly rows in two islands — 2026-02-20 17:00 →
    2026-03-31 23:00 (943 h) and 2026-08-01 00:00 → 2026-09-24 07:00 (1,304 h) — with a
    **122-day hole** between them and nothing at all before 2026-02-20. Against the
    meters' now much longer span it covers only **14.6 %** of the hourly slots (it was
    23.3 % before the mySET history was merged in). Inside each island it is 100 %
    complete, and it carries ~2 days of forecast, so it is a fine *serving* source and a
    useless *training* source. But the tables that feed it are complete:
    `ds_dev_silver.om_weather_hourly` and `raw.om_weather_features_meters` both hold all
    5,175 hours from 2026-02-20 17:00 to 2026-09-24 07:00, with daily `_sdc_extracted_at`
    stamps from 2026-04-14 to 2026-09-22. The range missing from gold (2026-04-01 00:00
    → 2026-07-31 23:00, 2,928 h) is cut on calendar-month boundaries and splits single
    daily batches, so it was **deleted from gold**, not lost upstream. The gold model is
    incremental on `_sdc_extracted_at`, so it will never re-import those rows by itself:
    a `dbt run --full-refresh` of that model, or a one-off insert from raw, restores
    them.

13. **Open-Meteo ICON-D2 covers 100 % of the meter span, with no hole.** The window the
    notebook asks for now starts at **2024-12-31**, because that is where the mySET
    history begins: 15,169 hourly rows to 2026-09-24, 100 % hourly completeness, no gap
    larger than an hour and **no nulls in any column**. The download asks for the same
    model as the pipeline (`models=icon_d2`) and takes the history from Open-Meteo's
    Historical Forecast API, which has no unfinalised tail. The 18-day temperature hole
    an earlier extract carried (2026-06-22 → 2026-07-08 21:00) was a **package bug**,
    not an archive limitation: `download_raw_weather` let the forecast frame override
    the archive over their 92-day overlap, and the forecast API returns all-null rows
    for the oldest ~17 days of a 92-day `past_days` window. Worse,
    `build_weather_features` zero-filled radiation and cloud cover in that window, so
    that extract carried 17 days of fake zero irradiance on top of the missing
    temperature. Fixed by letting history win on overlap and dropping all-null rows
    instead of zero-filling them.

14. **Where the two sources measure the same thing, they agree exactly.** Over the
    comparison week (2026-08-05 → 2026-08-12) `shortwave_radiation`, `temperature_2m`
    and `cloud_cover` all come out at correlation **1.000 with a mean absolute
    difference of 0.00** — the gold table and the fresh download are the same ICON-D2
    numbers. Two things had to be right before that was true. The download must **not**
    force an `elevation`: the tap sends none, so Open-Meteo uses its 90 m DEM (1172 m
    here), and forcing 1100 m downscales temperature by lapse rate over the 72 m
    difference, ≈0.47 °C — the constant 0.45 °C offset (std 0.06) this notebook used to
    report. And it must ask for `icon_d2` rather than falling back to the ERA5 archive
    (~31 km) for anything older than 92 days. What remains is a **column-definition**
    difference rather than a model one: `global_tilted_irradiance` correlates 0.997 but
    means 272.48 W/m² in gold against 285.85 from the package, because the tap requests
    no tilt or azimuth (so the gold column is plain horizontal irradiance under a tilted
    name) while the package asks for tilt 30° / azimuth 0. Where the definitions differ
    outright the sources do not agree at all: `effective_solar_pv` means 265.08 W/m² in
    gold against 0.32 from the package (a clipped cosine), and `cloud_cover_diff` is
    absolute in one (mean 20.82) and signed in the other (mean 0.60), correlation 0.077.
    **The two sources are not interchangeable column by column**, and the gold table
    additionally stores `solar_elevation` on a fixed UTC+1 clock (mean absolute error
    2.50° against the site's true geometry, versus 4.90° read as UTC and 6.93° read as a
    DST-aware local clock, and a peak bin that stays at 12:00 in both islands while the
    radiation peak moves from 13:00 to 14:00), an unclipped range down to −55°, a
    `theoretical_prod` in the 10⁵ range and an `is_daylight` that disagrees with its own
    `solar_elevation > 0` on 11 % of rows.

15. **Weather explains export, not import.** The correlation heatmap above is computed
    on the daylight hours of the exporting devices, and now includes their mySET
    history: `grid_export` lines up with `shortwave_radiation`,
    `global_tilted_irradiance`, `effective_solar_pv` and `solar_elevation` and runs
    against `cloud_cover`, while `grid_import` stays close to zero on every feature.
    (The exact coefficients are the ones annotated in that heatmap; they are not printed
    as text.) For a single mature PV device the export/irradiance relation is clearly
    monotone over the 4,325 daylight hours plotted, with a floor of zero-export hours
    where self-consumption absorbs the whole output.

### Open questions for notebook 02

* **Late joiners.** 26 of 41 devices arrive from 2026-05-21 on and so have at most 124
  days; the shortest span in the cohort is 12.3 days, well under the 42-day sufficiency
  bar. Drop them, train them with a shorter CV, or pool them into a global model? The
  mySET merge moves five devices out of this group and into the mature one, but it does
  not change the question for the remaining 26.
* **Can the two PV-frame devices' pre-May history be trained on?** `c2g-9FFB8A0D0` and
  `c2g-9FFB8AA78` are the two mySET devices whose readings do **not** reconcile with the
  MQTT meter over the overlap (finding 11): mySET exports more and imports almost
  nothing. Their 48,533 and 42,485 pre-May rows are therefore of unknown quality, while
  the other three devices' are well validated. Notebook 02 has to decide whether to use
  them, use only their export, or restrict those two to their MQTT history.
* **A two-hour hole on 2025-10-26 that the grid will not fill.** The mySET rows for the
  ambiguous fall-back hour are gone (finding 10), so the five devices are missing
  00:15 → 02:00 UTC that night. `max_gap_hours` is **1** in the default config, so
  `build_regular_grid` will flag those hours rather than interpolate them — which is
  probably right, but it should be a deliberate choice and not a surprise.
* **Stalled devices.** Eight devices stop well before the extract date (the earliest in
  June 2026). Decommissioned, or a broken ingestion? They should not be judged by
  "days since last reading" without knowing which.
* **The gap flag is inflated by construction.** `build_regular_grid` reindexes *every*
  device onto the *global* hourly range, so a June 2026 device now carries nineteen
  months of `gap_flag = True` and the overall gap share reads **65.8 %** (407,847 of
  619,633 rows) — higher than before precisely because the global range got longer.
  Per-device reindexing (or trimming to each device's own span) would make coverage
  statistics mean what they appear to mean.
* **Which weather source, and when.** Training has to use Open-Meteo; serving may well
  read the gold table. Given finding 14, that combination silently feeds the model
  differently-defined features at inference time. Either the gold table is
  reconciled to the package definitions, or serving downloads from Open-Meteo too.
* **Upstream and package to-do list.** Four fixes belong outside this notebook: restore
  the deleted gold weather range with a `dbt run --full-refresh` of
  `om_weather_features_meters`; add tilt and azimuth to the tap's Open-Meteo config so
  that `global_tilted_irradiance` is actually tilted; make the silver dedup key
  `(device_id, ts, cf_type)` — or validate the payload's `Meter` against its `Type` — so
  that a mislabelled message stops evicting a genuine PV reading; and give the package's
  dataset config a per-source **`ts_offset`**, so the mySET +15 min does not have to live
  in a notebook cell. The mySET tail (2026-06-04 → 06-08) is worth raising with whoever
  produces the portal exports as well.
* **The DST hour.** Both directions are now accounted for: the autumn fall-back shows up
  as 36 duplicates on the MQTT side and as the two-hour hole on the mySET side, and the
  spring forward-jump shows up in the mySET export as 92-row days (2025-03-30,
  2026-03-29) with a single 75-minute step. What is *not* checked is whether the MQTT
  side loses an hour on those same spring nights.
* **Zero-export hours at zero irradiance.** The export/irradiance scatter has a
  vertical stripe of non-zero export at exactly 0 W/m² tilted irradiance during
  flagged daylight hours — worth confirming it is the dawn/dusk labelling of the
  hourly radiation and not a misalignment.
* **A device that never imports.** One device shows a mean import of exactly 0.0000
  kWh/h over its whole history. For a meter at the grid connection point that is
  implausible and should be verified against the M2 side before it is trained on.
